# QLoRA 実機微調整ノートブック

小型 LLM（TinyLlama-1.1B）を **QLoRA で実際に微調整**する。
**Colab（無料T4 16GB）/ 12GB以上のGPUで動作**。`lora_qlora.md` の §6〜§9 を手で再現する。

流れ：4bitロード → LoRA装着 → SFT学習 → 推論比較 → アダプタ保存 → マージ。

> このノートはGPU前提。CPU/Jetsonで概念だけ確認したいなら `lora_qlora_demo.ipynb` を先に。

## 0. 環境セットアップ

In [ ]:
# バージョンを固定（APIの互換性のため）。Colabなら実行、ローカルは適宜。
%pip install -q -U "transformers>=4.44" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.43" "datasets>=2.20" "accelerate>=0.33"

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPUが必要です（Colabなら ランタイム→GPU）'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
# bf16が使えるか（Ampere以降）。T4はfp16。
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('compute_dtype =', compute_dtype)

---

## 1. 4bit（NF4）でベースモデルをロード

`lora_qlora.md` §5〜§6 の設定。`nf4 + double_quant + compute_dtype` が QLoRA の標準。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'   # ungated・軽量。Qwen/Qwen2.5-0.5B-Instruct でも可

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',            # NormalFloat4（§5.2）
    bnb_4bit_use_double_quant=True,       # Double Quantization（§5.3）
    bnb_4bit_compute_dtype=compute_dtype, # 計算は16bit（§6.1）
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map='auto',
)
model.config.use_cache = False  # 学習時はoff（gradient checkpointと両立）
print('4bitロード後のVRAM:', round(torch.cuda.memory_allocated()/1e9,2), 'GB')

---

## 2. LoRA を装着

§7.2 の手順。`prepare_model_for_kbit_training` を忘れないこと（§11-1）。

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)  # 量子化モデルの学習準備（必須）

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,                 # α/r = 2（§2.4）
    target_modules='all-linear',   # 全Linear（QLoRA論文推奨, §3）
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 例: trainable params: ~12M || all params: ~1.1B || trainable%: ~1.1%

---

## 3. データセット

QLoRA デモの定番 `mlabonne/guanaco-llama2-1k`（1000件のチャット）。`text` 列をそのままSFTに使う。

In [ ]:
from datasets import load_dataset
dataset = load_dataset('mlabonne/guanaco-llama2-1k', split='train')
print(dataset)
print('--- サンプル ---')
print(dataset[0]['text'][:500])

---

## 4. 学習（SFT）

`trl` の SFTTrainer。学習率は Full FT より高め 2e-4（§8）。optimizer は paged 8bit Adam（§5.4）。
デモなので 60 step だけ回す（本番は num_train_epochs で）。

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir='./qlora-tinyllama',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # 実効バッチ=8
    max_steps=60,                       # デモ用。本番は num_train_epochs=1〜3
    learning_rate=2e-4,                 # LoRAは高めでOK（§8）
    bf16=(compute_dtype==torch.bfloat16),
    fp16=(compute_dtype==torch.float16),
    logging_steps=10,
    optim='paged_adamw_8bit',           # paged optimizer（§5.4）
    gradient_checkpointing=True,        # 活性化メモリ削減（§11-3）
    max_seq_length=512,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=args,
    processing_class=tokenizer,
)
trainer.train()
print('学習中ピークVRAM:', round(torch.cuda.max_memory_allocated()/1e9,2), 'GB')

---

## 5. 推論で確認

学習したアダプタ付きで生成してみる。

In [ ]:
def generate(prompt, max_new_tokens=200):
    text = f'<s>[INST] {prompt} [/INST]'
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    model.config.use_cache = True
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=True, temperature=0.7, top_p=0.9)
    model.config.use_cache = False
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(generate('What is LoRA in machine learning? Answer concisely.'))

---

## 6. アダプタの保存

§9。保存されるのは**アダプタだけ**（数十MB）。ベースは別管理。

In [ ]:
model.save_pretrained('./qlora-tinyllama-adapter')
tokenizer.save_pretrained('./qlora-tinyllama-adapter')
import os
sz = sum(os.path.getsize(os.path.join('./qlora-tinyllama-adapter',f))
         for f in os.listdir('./qlora-tinyllama-adapter'))
print('保存サイズ:', round(sz/1e6,1), 'MB （ベース1.1Bに対してこれだけ）')

---

## 7. マージして単体モデルにする（任意・本番デプロイ用）

§9.1。**4bitのままマージは不可** → fp16でベースを再ロードしてからマージするのが安全。
（VRAMに余裕がなければスキップ可）

In [ ]:
# 推論レイテンシをゼロにしたい場合のマージ手順（メモリに注意）
from peft import PeftModel

base_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base_fp16, './qlora-tinyllama-adapter')
merged = merged.merge_and_unload()   # W = W0 + (alpha/r) B A を焼き込む
merged.save_pretrained('./qlora-tinyllama-merged')
tokenizer.save_pretrained('./qlora-tinyllama-merged')
print('マージ完了 → ./qlora-tinyllama-merged （LoRA分岐が消え追加レイテンシ0）')

---

## 8. 手を動かす課題

理解を定着させるために試す：

1. **r を変える**: `r=8` と `r=64` で `print_trainable_parameters()` と最終lossを比較。
2. **target_modules を絞る**: `['q_proj','v_proj']` だけにして性能・パラメータ数の差を見る（§3）。
3. **量子化をoffに**: `load_in_4bit=False` の素のLoRA（§4のメモリ）とVRAMを比較。
4. **alpha の効果**: `lora_alpha` を 16 / 64 に変えて生成品質の変化を観察。
5. **DoRA を試す**: `LoraConfig(..., use_dora=True)` にして比較（§10）。
6. **自分のデータ**: `text` 列を持つ自前データセットに差し替えてドメイン特化させる。

> 観察したことは #daily-action-log に記録すると、デイリーレビューの「学習・成長」に反映される。